# Step 3 — Preprocesamiento con GenericTransformer

**Estrategia para evitar OOM (136M filas, 3 GB):**

1. Fit del transformer en una muestra estratificada del train (todos los positivos + N negativos)
2. Transform completo por batches con `pq.ParquetWriter`
3. Output: `data/processed/train.parquet`, `val.parquet`, `test.parquet`

**La columna `split` ya existe en el dataset** — la usamos directamente, no recalculamos.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')

In [ ]:
import json
import joblib
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from transformers.generic_transformer import GenericTransformer

project_folder = Path(config['project_folder'])
raw_path = project_folder / config['data']['raw']['local_path'] / 'features.parquet'
processed_dir = project_folder / config['data']['processed']['local_path']
processed_dir.mkdir(parents=True, exist_ok=True)

trans_cfg = json.loads((project_folder / 'src' / 'config' / 'transformations.json').read_text())
target_col = config['model']['objective_column']  # fire_occurred
cols_to_drop = config['data']['columns_to_drop'] + ['split']  # date, cell_id, split

pf = pq.ParquetFile(raw_path)
print(f'Dataset: {pf.metadata.num_rows:,} filas, {len(pf.schema_arrow)} columnas')

## 3.1 Crear muestra estratificada del train para fit

Incluimos **todos los positivos** del train (≈54K) + **muestra de negativos** (≈2M).
Las estadísticas (median, std) son estables con 2M filas.

In [ ]:
NEGATIVES_PER_BATCH = 15_000  # ~2M negativos total con 137 row groups

train_positives = []
train_negatives = []

for batch in pf.iter_batches(batch_size=1_000_000, columns=None):
    df_batch = batch.to_pandas()
    train_mask = df_batch['split'] == 'train'
    train_batch = df_batch[train_mask]
    
    if len(train_batch) == 0:
        continue
    
    pos = train_batch[train_batch[target_col] == True]
    neg = train_batch[train_batch[target_col] == False]
    
    train_positives.append(pos)
    if len(neg) > NEGATIVES_PER_BATCH:
        train_negatives.append(neg.sample(n=NEGATIVES_PER_BATCH, random_state=42))
    else:
        train_negatives.append(neg)

train_sample = pd.concat(train_positives + train_negatives, ignore_index=True)
del train_positives, train_negatives

n_pos = train_sample[target_col].sum()
n_neg = len(train_sample) - n_pos
print(f'Muestra del train: {len(train_sample):,} filas')
print(f'  Positivos: {n_pos:,} ({n_pos/len(train_sample)*100:.2f}%)')
print(f'  Negativos: {n_neg:,} ({n_neg/len(train_sample)*100:.2f}%)')

## 3.2 Preparar muestra: conversiones y drops

In [ ]:
# Convertir is_protected_area bool → int
if 'is_protected_area' in train_sample.columns:
    train_sample['is_protected_area'] = train_sample['is_protected_area'].astype(int)

# Dropear columnas que no son features
X_sample = train_sample.drop(columns=cols_to_drop + [target_col], errors='ignore')
y_sample = train_sample[target_col]

print(f'Features para fit: {X_sample.shape[1]}')
print(f'Columnas: {list(X_sample.columns)}')

## 3.3 Fit del GenericTransformer en la muestra del train

In [ ]:
# Filtrar transformations.json a solo las columnas que existen
trans_cfg_filtered = {k: v for k, v in trans_cfg.items() if k in X_sample.columns}
missing_from_cfg = [k for k in trans_cfg.keys() if k not in X_sample.columns]

if missing_from_cfg:
    print(f'Columnas en transformations.json pero no en dataset (se construyen en Step 5):')
    print(f'  {missing_from_cfg}')

transformer = GenericTransformer(trans_cfg_filtered)
transformer.fit(X_sample, y_sample)

logger.info('Transformer fitted on %d features', len(trans_cfg_filtered))

In [ ]:
# Verificar transform en la muestra
X_transformed = transformer.transform(X_sample)
print(f'Shape después de transform: {X_transformed.shape}')
print(f'Nulos después de transform: {X_transformed.isnull().sum().sum()}')

## 3.4 Serializar transformer

In [ ]:
transformer_path = processed_dir / 'transformer.joblib'
joblib.dump(transformer, transformer_path)
logger.info('Transformer guardado en %s', transformer_path)

## 3.5 Transform completo por batches y guardar splits

Procesamos el dataset completo por batches y escribimos cada split a su archivo.

In [ ]:
def get_feature_columns(schema):
    """Obtener lista de columnas feature (excluyendo drops y target)"""
    all_cols = [f.name for f in schema]
    exclude = set(cols_to_drop + [target_col])
    return [c for c in all_cols if c not in exclude]

feature_cols = get_feature_columns(pf.schema_arrow)
print(f'Feature columns: {len(feature_cols)}')

In [ ]:
from tqdm.auto import tqdm

split_stats = {'train': {'rows': 0, 'positives': 0},
               'val': {'rows': 0, 'positives': 0},
               'test': {'rows': 0, 'positives': 0}}

writers = {}
output_schema = None

try:
    for batch in tqdm(pf.iter_batches(batch_size=1_000_000), 
                      total=pf.metadata.num_row_groups,
                      desc='Processing batches'):
        df_batch = batch.to_pandas()
        
        # Convertir is_protected_area
        if 'is_protected_area' in df_batch.columns:
            df_batch['is_protected_area'] = df_batch['is_protected_area'].astype(int)
        
        for split_name in ['train', 'val', 'test']:
            split_mask = df_batch['split'] == split_name
            split_df = df_batch[split_mask]
            
            if len(split_df) == 0:
                continue
            
            # Extraer features y target
            X_batch = split_df[feature_cols].copy()
            y_batch = split_df[target_col].values
            
            # Transform
            X_transformed = transformer.transform(X_batch)
            X_transformed[target_col] = y_batch
            
            # Stats
            split_stats[split_name]['rows'] += len(X_transformed)
            split_stats[split_name]['positives'] += y_batch.sum()
            
            # Escribir
            table = pa.Table.from_pandas(X_transformed, preserve_index=False)
            
            if split_name not in writers:
                out_path = processed_dir / f'{split_name}.parquet'
                writers[split_name] = pq.ParquetWriter(out_path, table.schema)
            
            writers[split_name].write_table(table)

finally:
    for w in writers.values():
        w.close()

print('\nTransform completo.')

## 3.6 Verificar resultados

In [ ]:
print('Estadísticas por split:')
print(f'{"Split":<8} {"Filas":>15} {"Positivos":>12} {"% Positivos":>12}')
print('-' * 50)

for split_name, stats in split_stats.items():
    pct = stats['positives'] / stats['rows'] * 100 if stats['rows'] > 0 else 0
    print(f'{split_name:<8} {stats["rows"]:>15,} {stats["positives"]:>12,} {pct:>11.4f}%')

total_rows = sum(s['rows'] for s in split_stats.values())
total_pos = sum(s['positives'] for s in split_stats.values())
print('-' * 50)
print(f'{"TOTAL":<8} {total_rows:>15,} {total_pos:>12,} {total_pos/total_rows*100:>11.4f}%')

In [ ]:
# Verificar archivos generados
for split_name in ['train', 'val', 'test']:
    path = processed_dir / f'{split_name}.parquet'
    pf_check = pq.ParquetFile(path)
    size_mb = path.stat().st_size / 1e6
    print(f'{split_name}.parquet: {pf_check.metadata.num_rows:,} filas, {size_mb:.1f} MB')

In [ ]:
# Leer muestra del train transformado para inspección
train_check = pq.read_table(processed_dir / 'train.parquet', 
                            columns=None).slice(0, 5).to_pandas()
print(f'Columnas en train.parquet: {list(train_check.columns)}')
train_check.head()

## Resumen

- GenericTransformer fitted en muestra estratificada del train (todos los positivos + ~2M negativos)
- Transform aplicado al dataset completo por batches (sin OOM)
- Archivos generados:
  - `data/processed/transformer.joblib`
  - `data/processed/train.parquet`
  - `data/processed/val.parquet`
  - `data/processed/test.parquet`

**Siguiente paso**: `step4_exploration_transformed.ipynb`